In [61]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import pprint

In [62]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

Device: cuda


## Configuration

In [63]:
D_MODEL = 128            # Embedding size
BLOCK_SIZE = 32
BATCH_SIZE = 64

## Dataset

In [64]:
data_path = Path("../../datasets/tinyshakespeare.txt")
text = data_path.read_text(encoding="utf-8")
chars = sorted(set(text))
VOCAB_SIZE = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]


def decode(ids):
    return "".join(itos[i] for i in ids)



data = torch.tensor(encode(text), dtype=torch.long)

print("Characters:", len(text))
print(pprint.pprint(stoi, compact=True))
print(f"VOCAB_SIZE = {VOCAB_SIZE}")
print("Encoded shape:", data)    

Characters: 1115393
{'\n': 0,
 ' ': 1,
 '!': 2,
 '$': 3,
 '&': 4,
 "'": 5,
 ',': 6,
 '-': 7,
 '.': 8,
 '3': 9,
 ':': 10,
 ';': 11,
 '?': 12,
 'A': 13,
 'B': 14,
 'C': 15,
 'D': 16,
 'E': 17,
 'F': 18,
 'G': 19,
 'H': 20,
 'I': 21,
 'J': 22,
 'K': 23,
 'L': 24,
 'M': 25,
 'N': 26,
 'O': 27,
 'P': 28,
 'Q': 29,
 'R': 30,
 'S': 31,
 'T': 32,
 'U': 33,
 'V': 34,
 'W': 35,
 'X': 36,
 'Y': 37,
 'Z': 38,
 'a': 39,
 'b': 40,
 'c': 41,
 'd': 42,
 'e': 43,
 'f': 44,
 'g': 45,
 'h': 46,
 'i': 47,
 'j': 48,
 'k': 49,
 'l': 50,
 'm': 51,
 'n': 52,
 'o': 53,
 'p': 54,
 'q': 55,
 'r': 56,
 's': 57,
 't': 58,
 'u': 59,
 'v': 60,
 'w': 61,
 'x': 62,
 'y': 63,
 'z': 64}
None
VOCAB_SIZE = 65
Encoded shape: tensor([18, 47, 56,  ..., 52, 45,  8])


## Tokenizer

## Simple model

In [65]:
model = nn.Sequential(
    nn.Embedding(VOCAB_SIZE, D_MODEL),
    nn.Linear(D_MODEL, VOCAB_SIZE)
)

# -------------------------
# Model
# -------------------------
class GPT(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(len(chars), D_MODEL)

        self.Wq = nn.Linear(D_MODEL, D_MODEL)
        self.Wk = nn.Linear(D_MODEL, D_MODEL)
        self.Wv = nn.Linear(D_MODEL, D_MODEL)

        self.lm_head = nn.Linear(D_MODEL, len(chars))

    def forward(self, x):

        # [B, T] -> [B, T, 128]
        x = self.embedding(x)

        # Q, K, V
        q = self.Wq(x)
        k = self.Wk(x)
        v = self.Wv(x)

        # Attention scores
        scores = q @ k.transpose(-2, -1)

        scores = scores / (D_MODEL ** 0.5)

        # Causal mask
        mask = torch.tril(
            torch.ones(BLOCK_SIZE, BLOCK_SIZE,device=x.device)
        )

        scores = scores.masked_fill(
            mask[:x.size(1), :x.size(1)] == 0,
            float("-inf")
        )

        # Attention weights
        weights = torch.softmax(scores, dim=-1)

        # Weighted values
        x = weights @ v

        # Predict next character
        logits = self.lm_head(x)

        return logits






## Training

In [66]:
model = GPT().to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

running = []
for step in range(2000):
    

    starts = torch.randint(
        0, 
        len(data) - BLOCK_SIZE - 1, 
        (BATCH_SIZE,) 
    ).to(DEVICE)

    x = torch.stack([
        data[i:i + BLOCK_SIZE] 
        for i in starts 
    ]).to(DEVICE)

    y = torch.stack([
        data[i + 1:i + BLOCK_SIZE + 1] 
        for i in starts 
    ]).to(DEVICE)

   # print(starts,x,y)
    
    # # print("x:", decode(x.tolist()))
    # # print("y:", decode(y.tolist()))


    logits = model(x)

    loss = F.cross_entropy( logits.reshape(-1, len(chars)), y.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running.append(loss.item())
    if step % 100 == 0:
        print(step, sum(running[-100:]) / len(running[-100:]))
    

0 4.193157196044922
100 3.0534987258911133
200 2.650830726623535
300 2.5698691821098327
400 2.516194260120392
500 2.4774803948402404
600 2.4689097905159
700 2.4544578433036803
800 2.4412148427963256
900 2.44131618976593
1000 2.4352171134948732
1100 2.4341196990013123
1200 2.4332814025878906
1300 2.4290239906311033
1400 2.432520034313202
1500 2.4238021111488344
1600 2.4256075501441954
1700 2.4212327647209166
1800 2.4266855716705322
1900 2.420419557094574


In [67]:
# -------------------------
# Generate
# -------------------------

x = torch.tensor(
    [stoi["A"]],
    device=DEVICE
)

model.eval()

with torch.no_grad():

    for _ in range(500):

        # x[-1:] has shape [1]
        logits = model(x[-1:])

        # If logits is [1, vocab_size],
        # take the last prediction
        logits = logits[-1]

        probs = torch.softmax(
            logits,
            dim=-1
        )

        # multinomial gives [1]
        next_token = torch.multinomial(
            probs,
            1
        )

        # Both are now [1]
        x = torch.cat([
            x,
            next_token
        ])

print("\nGenerated:\n")
print("".join(itos[i.item()] for i in x))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x32 and 1x128)